# UK House Price Intelligence — Real HM Land Registry Data

**2026 portfolio edition · Real transactions · SQL + ML + uncertainty + explainability**

[Open in Google Colab](https://colab.research.google.com/github/Jorgoluka100/uni_projects/blob/main/01_UK_House_Price_Analysis_and_Prediction.ipynb)

This project analyses **real HM Land Registry Price Paid Data** for England and Wales and builds a forward-looking residential sale-price model.

> **Current data snapshot:** the HM Land Registry Price Paid Data page was updated **29 September 2026** and currently includes **August 2026** data.

> **Evaluation discipline:** HM Land Registry warns that the two most recent months are incomplete because registration can lag a sale by weeks or months. The model therefore evaluates on **January–June 2026** only. July–August 2026 are loaded for provisional drift monitoring, not headline model scoring.

### What this demonstrates for 2026 data roles
- official real-world government data
- DuckDB SQL analytics over large CSV files
- explicit data-quality and licensing checks
- temporal train / validation / future-test design
- leakage-safe feature engineering
- strong geographic baseline vs CatBoost
- MAE, RMSE, R², MAPE and within-20% accuracy
- validation-calibrated conformal prediction intervals
- CatBoost SHAP explanations
- segment-level error analysis
- provisional market/drift monitoring
- serialized-model reload and inference profiling
- automated integrity checks

**Source:** HM Land Registry Price Paid Data  
**Licence attribution:** Contains HM Land Registry data © Crown copyright and database right 2021. This data is licensed under the Open Government Licence v3.0.

This notebook is an analytical prototype, not a surveyor valuation, mortgage decision or investment recommendation.


## 1. Setup and reproducibility

In [ ]:
import importlib.util
import subprocess
import sys
import json
import os
import random
import time
import urllib.request
from pathlib import Path

required = {
    "duckdb": "duckdb>=1.3.2",
    "catboost": "catboost>=1.2.8",
}

missing = [
    package
    for package in required
    if importlib.util.find_spec(package) is None
]

if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *[required[p] for p in missing],
    ])

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd() / ".cache" / "uk_house_price_project"
ROOT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)

print("Python:", sys.version.split()[0])
print("DuckDB:", duckdb.__version__)
print("Seed:", SEED)
print("Working directory:", ROOT)


## 2. Download the official 2025–2026 Price Paid Data

The yearly files are updated as registrations arrive. Rather than hard-coding an expected row count, the notebook records the observed source snapshot and validates the schema/date range.

Exact address fields are read because they are present in the official file, but the model uses only **coarse geography** and never exposes house numbers, street names or locality in its feature set.


In [ ]:
URLS = {
    2025: "https://price-paid-data.publicdata.landregistry.gov.uk/pp-2025.csv",
    2026: "https://price-paid-data.publicdata.landregistry.gov.uk/pp-2026.csv",
}

paths = {}

for year, url in URLS.items():
    path = ROOT / f"pp-{year}.csv"
    paths[year] = path

    if not path.exists():
        print("Downloading:", url)
        urllib.request.urlretrieve(url, path)

    print(
        year,
        f"{path.stat().st_size / 1e6:.1f} MB",
    )

con = duckdb.connect(database=":memory:")

file_list = ",".join(
    repr(str(path))
    for path in paths.values()
)

con.execute(f"""
CREATE VIEW price_paid AS
SELECT
    column00 AS transaction_id,
    CAST(column01 AS BIGINT) AS price,
    CAST(
        strptime(column02, '%Y-%m-%d %H:%M')
        AS DATE
    ) AS transfer_date,
    NULLIF(trim(column03), '') AS postcode,
    column04 AS property_type,
    column05 AS old_new,
    column06 AS duration,
    column07 AS paon,
    column08 AS saon,
    column09 AS street,
    column10 AS locality,
    column11 AS town_city,
    column12 AS district,
    column13 AS county,
    column14 AS ppd_category,
    column15 AS record_status
FROM read_csv(
    [{file_list}],
    header=false,
    all_varchar=true
)
""")

source_audit = con.execute("""
SELECT
    COUNT(*) AS raw_rows,
    COUNT(DISTINCT transaction_id) AS unique_ids,
    MIN(transfer_date) AS min_date,
    MAX(transfer_date) AS max_date,
    SUM(postcode IS NULL) AS missing_postcodes,
    MIN(price) AS min_price,
    MAX(price) AS max_price
FROM price_paid
""").df()

display(source_audit.T)

assert int(source_audit["raw_rows"].iloc[0]) > 500_000
assert (
    int(source_audit["raw_rows"].iloc[0])
    == int(source_audit["unique_ids"].iloc[0])
)
assert pd.Timestamp(
    source_audit["max_date"].iloc[0]
) >= pd.Timestamp("2026-06-30")

print("Source snapshot checks passed.")


## 3. Data contract and modelling population

The model focuses on ordinary residential transactions:

- detached, semi-detached, terraced and flats
- Category A standard price-paid transactions
- £20,000–£5,000,000
- non-missing postcode
- sale date from January 2025 onward

The target is the registered sale price. Exact address fields and transaction IDs are excluded from the modelling feature contract.


In [ ]:
con.execute("""
CREATE VIEW residential AS
SELECT
    transaction_id,
    price,
    transfer_date,
    postcode,
    split_part(postcode, ' ', 1) AS postcode_district,
    regexp_extract(postcode, '^[A-Z]+', 0) AS postcode_area,
    property_type,
    old_new,
    duration,
    town_city,
    district,
    county,
    EXTRACT(year FROM transfer_date)::INTEGER AS transfer_year,
    EXTRACT(month FROM transfer_date)::INTEGER AS transfer_month,
    EXTRACT(quarter FROM transfer_date)::INTEGER AS transfer_quarter,
    sin(
        2 * pi() * EXTRACT(month FROM transfer_date) / 12
    ) AS month_sin,
    cos(
        2 * pi() * EXTRACT(month FROM transfer_date) / 12
    ) AS month_cos
FROM price_paid
WHERE
    property_type IN ('D', 'S', 'T', 'F')
    AND ppd_category = 'A'
    AND price BETWEEN 20000 AND 5000000
    AND postcode IS NOT NULL
    AND transfer_date >= DATE '2025-01-01'
""")

quality = con.execute("""
SELECT
    COUNT(*) AS accepted_rows,
    MIN(transfer_date) AS first_sale,
    MAX(transfer_date) AS latest_sale,
    COUNT(DISTINCT postcode_district) AS postcode_districts,
    COUNT(DISTINCT county) AS counties
FROM residential
""").df()

display(quality.T)

raw_rows = int(source_audit["raw_rows"].iloc[0])
accepted_rows = int(quality["accepted_rows"].iloc[0])

print({
    "raw_rows": raw_rows,
    "accepted_rows": accepted_rows,
    "accepted_share": round(accepted_rows / raw_rows, 4),
})


## 4. SQL market intelligence

The first deliverable is useful even without ML: transaction volumes, medians, interquartile ranges and regional/property-type patterns.


In [ ]:
monthly = con.execute("""
SELECT
    date_trunc('month', transfer_date) AS sale_month,
    COUNT(*) AS transactions,
    median(price) AS median_price,
    avg(price) AS mean_price,
    quantile_cont(price, 0.25) AS p25,
    quantile_cont(price, 0.75) AS p75
FROM residential
GROUP BY 1
ORDER BY 1
""").df()

property_summary = con.execute("""
SELECT
    property_type,
    COUNT(*) AS transactions,
    median(price) AS median_price,
    quantile_cont(price, 0.25) AS p25,
    quantile_cont(price, 0.75) AS p75
FROM residential
GROUP BY 1
ORDER BY median_price DESC
""").df()

county_summary = con.execute("""
SELECT
    county,
    COUNT(*) AS transactions,
    median(price) AS median_price
FROM residential
WHERE transfer_date < DATE '2026-07-01'
GROUP BY 1
HAVING COUNT(*) >= 1000
ORDER BY median_price DESC
LIMIT 15
""").df()

display(property_summary)
display(county_summary)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
)

axes[0].plot(
    monthly["sale_month"],
    monthly["median_price"],
    marker="o",
)
axes[0].axvspan(
    pd.Timestamp("2026-07-01"),
    monthly["sale_month"].max(),
    alpha=0.15,
    label="provisional monitoring window",
)
axes[0].set_title("Median registered sale price")
axes[0].set_ylabel("Price (£)")
axes[0].legend()

axes[1].bar(
    property_summary["property_type"],
    property_summary["median_price"],
)
axes[1].set_title("Median price by property type")
axes[1].set_ylabel("Price (£)")

plt.tight_layout()
plt.show()


## 5. Leakage-safe feature table and temporal split

The evaluation mimics a real future deployment:

- **Train:** January–September 2025
- **Validation:** October–December 2025
- **Untouched test:** January–June 2026
- **Provisional monitoring only:** July–August 2026 (or any later rows already present in the yearly file)

No test or provisional rows influence fitting, thresholding or interval width.


In [ ]:
model_df = con.execute("""
SELECT
    price,
    transfer_date,
    postcode_district,
    postcode_area,
    property_type,
    old_new,
    duration,
    town_city,
    district,
    county,
    transfer_year,
    transfer_month,
    transfer_quarter,
    month_sin,
    month_cos
FROM residential
ORDER BY transfer_date, transaction_id
""").df()

categorical = [
    "postcode_district",
    "postcode_area",
    "property_type",
    "old_new",
    "duration",
    "town_city",
    "district",
    "county",
]

numeric = [
    "transfer_year",
    "transfer_month",
    "transfer_quarter",
    "month_sin",
    "month_cos",
]

features = categorical + numeric

for column in categorical:
    model_df[column] = (
        model_df[column]
        .fillna("__MISSING__")
        .astype(str)
    )

train = model_df[
    model_df["transfer_date"]
    < pd.Timestamp("2025-10-01")
].copy()

validation = model_df[
    (model_df["transfer_date"] >= pd.Timestamp("2025-10-01"))
    & (model_df["transfer_date"] < pd.Timestamp("2026-01-01"))
].copy()

test = model_df[
    (model_df["transfer_date"] >= pd.Timestamp("2026-01-01"))
    & (model_df["transfer_date"] < pd.Timestamp("2026-07-01"))
].copy()

provisional = model_df[
    model_df["transfer_date"] >= pd.Timestamp("2026-07-01")
].copy()

split_summary = pd.DataFrame({
    "split": [
        "train",
        "validation",
        "test",
        "provisional_monitoring",
    ],
    "rows": [
        len(train),
        len(validation),
        len(test),
        len(provisional),
    ],
    "from": [
        train["transfer_date"].min(),
        validation["transfer_date"].min(),
        test["transfer_date"].min(),
        provisional["transfer_date"].min()
        if len(provisional)
        else pd.NaT,
    ],
    "to": [
        train["transfer_date"].max(),
        validation["transfer_date"].max(),
        test["transfer_date"].max(),
        provisional["transfer_date"].max()
        if len(provisional)
        else pd.NaT,
    ],
})

display(split_summary)

assert train["transfer_date"].max() < validation["transfer_date"].min()
assert validation["transfer_date"].max() < test["transfer_date"].min()
assert test["transfer_date"].max() <= pd.Timestamp("2026-06-30")

print("Temporal split checks passed.")


## 6. Honest baselines

A strong model must beat a sensible valuation heuristic. The primary baseline uses the **training-set median price for each postcode district × property type**, with fallback to property-type and then global median.


In [ ]:
global_median = float(train["price"].median())

type_median = (
    train
    .groupby("property_type")["price"]
    .median()
)

district_type_median = (
    train
    .groupby([
        "postcode_district",
        "property_type",
    ])["price"]
    .median()
)

def baseline_predict(frame):
    keys = pd.MultiIndex.from_frame(
        frame[[
            "postcode_district",
            "property_type",
        ]]
    )

    local = np.asarray(
        district_type_median.reindex(keys),
        dtype=float,
    )

    fallback = (
        frame["property_type"]
        .map(type_median)
        .fillna(global_median)
        .to_numpy(float)
    )

    return np.where(
        np.isfinite(local),
        local,
        fallback,
    )

def regression_metrics(y_true, prediction):
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(prediction, dtype=float)

    absolute_error = np.abs(y - p)
    ape = absolute_error / y

    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": mean_squared_error(y, p) ** 0.5,
        "R2": r2_score(y, p),
        "MAPE_pct": ape.mean() * 100,
        "within_10pct": (ape <= 0.10).mean(),
        "within_20pct": (ape <= 0.20).mean(),
    }

baseline_validation = baseline_predict(validation)
baseline_test = baseline_predict(test)

baseline_results = pd.DataFrame({
    "Global median": regression_metrics(
        test["price"],
        np.repeat(global_median, len(test)),
    ),
    "District + property baseline": regression_metrics(
        test["price"],
        baseline_test,
    ),
}).T

display(baseline_results.round(3))


## 7. CatBoost valuation model

CatBoost is well suited to location-heavy tabular data because it can model categorical interactions without exploding thousands of postcode districts into a huge one-hot matrix.

The target is `log1p(price)`, reducing the influence of the long luxury-property tail. Early stopping uses only late-2025 validation data.


In [ ]:
MAX_TRAIN = 500_000
MAX_VALID = 120_000

def deterministic_cap(frame, n):
    if len(frame) <= n:
        return frame.copy()

    return frame.sample(
        n=n,
        random_state=SEED,
    ).copy()

train_fit = deterministic_cap(
    train,
    MAX_TRAIN,
)

validation_fit = deterministic_cap(
    validation,
    MAX_VALID,
)

cat_indices = [
    features.index(column)
    for column in categorical
]

train_pool = Pool(
    train_fit[features],
    np.log1p(train_fit["price"]),
    cat_features=cat_indices,
)

validation_pool = Pool(
    validation_fit[features],
    np.log1p(validation_fit["price"]),
    cat_features=cat_indices,
)

model = CatBoostRegressor(
    iterations=650,
    depth=9,
    learning_rate=0.07,
    loss_function="MAE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=SEED,
    allow_writing_files=False,
    thread_count=-1,
    verbose=100,
)

model.fit(
    train_pool,
    eval_set=validation_pool,
    early_stopping_rounds=70,
    use_best_model=True,
)

pred_validation = np.expm1(
    model.predict(validation[features])
).clip(20_000, 5_000_000)

pred_test = np.expm1(
    model.predict(test[features])
).clip(20_000, 5_000_000)

print("Best iteration:", model.get_best_iteration())
display(
    pd.DataFrame(
        [regression_metrics(
            validation["price"],
            pred_validation,
        )],
        index=["validation"],
    ).round(3)
)


## 8. Untouched 2026 evaluation

The headline result is measured only on January–June 2026 sales that were never used for model fitting or tuning.


In [ ]:
evaluation = pd.DataFrame({
    "Global median": regression_metrics(
        test["price"],
        np.repeat(global_median, len(test)),
    ),
    "District + property baseline": regression_metrics(
        test["price"],
        baseline_test,
    ),
    "CatBoost": regression_metrics(
        test["price"],
        pred_test,
    ),
}).T

baseline_mae = evaluation.loc[
    "District + property baseline",
    "MAE",
]
model_mae = evaluation.loc[
    "CatBoost",
    "MAE",
]

improvement = 1 - model_mae / baseline_mae

display(evaluation.round(3))
print(
    "MAE improvement vs strong baseline:",
    f"{improvement:.1%}",
)

comparison = pd.DataFrame({
    "actual": test["price"].to_numpy(),
    "prediction": pred_test,
    "property_type": test["property_type"].to_numpy(),
    "county": test["county"].to_numpy(),
})

comparison["absolute_error"] = np.abs(
    comparison["actual"]
    - comparison["prediction"]
)

comparison["absolute_pct_error"] = (
    comparison["absolute_error"]
    / comparison["actual"]
)

sample = comparison.sample(
    min(15_000, len(comparison)),
    random_state=SEED,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
)

axes[0].scatter(
    sample["actual"],
    sample["prediction"],
    s=5,
    alpha=0.20,
)

limit = min(
    1_500_000,
    np.quantile(sample["actual"], 0.995),
)

axes[0].plot(
    [0, limit],
    [0, limit],
    linestyle="--",
)
axes[0].set_xlim(0, limit)
axes[0].set_ylim(0, limit)
axes[0].set_xlabel("Actual (£)")
axes[0].set_ylabel("Predicted (£)")
axes[0].set_title("2026 future-test predictions")

property_error = (
    comparison
    .groupby("property_type")["absolute_pct_error"]
    .median()
    .sort_values()
)

axes[1].bar(
    property_error.index,
    property_error.values,
)
axes[1].set_title("Median percentage error by property type")
axes[1].set_ylabel("Median absolute percentage error")

plt.tight_layout()
plt.show()


## 9. Validation-calibrated conformal intervals

A point estimate hides uncertainty. The notebook learns a **90% absolute-residual interval width from validation data only**, separately by property type where enough observations exist, with a global fallback.

This is a simple split-conformal style interval—not a claim of perfect conditional coverage.


In [ ]:
validation_residual = np.abs(
    validation["price"].to_numpy()
    - pred_validation
)

GLOBAL_Q90 = float(
    np.quantile(
        validation_residual,
        0.90,
        method="higher",
    )
)

validation_interval_frame = pd.DataFrame({
    "property_type": validation["property_type"].to_numpy(),
    "residual": validation_residual,
})

type_q90 = (
    validation_interval_frame
    .groupby("property_type")["residual"]
    .quantile(0.90)
)

interval_width = (
    test["property_type"]
    .map(type_q90)
    .fillna(GLOBAL_Q90)
    .to_numpy(float)
)

lower = np.maximum(
    20_000,
    pred_test - interval_width,
)

upper = np.minimum(
    5_000_000,
    pred_test + interval_width,
)

covered = (
    (test["price"].to_numpy() >= lower)
    & (test["price"].to_numpy() <= upper)
)

interval_report = pd.DataFrame([{
    "nominal_coverage": 0.90,
    "test_coverage": covered.mean(),
    "median_interval_width": np.median(upper - lower),
    "mean_interval_width": np.mean(upper - lower),
    "global_validation_q90": GLOBAL_Q90,
}])

display(interval_report.round(3))

coverage_by_type = (
    pd.DataFrame({
        "property_type": test["property_type"].to_numpy(),
        "covered": covered,
        "width": upper - lower,
    })
    .groupby("property_type")
    .agg(
        rows=("covered", "size"),
        coverage=("covered", "mean"),
        median_width=("width", "median"),
    )
)

display(coverage_by_type.round(3))


## 10. SHAP explainability

CatBoost can calculate SHAP values directly. The chart below uses a deterministic sample from the **future test set**. Attribution is predictive—not causal.


In [ ]:
SHAP_ROWS = min(
    5_000,
    len(test),
)

shap_sample = test.sample(
    n=SHAP_ROWS,
    random_state=SEED,
)

shap_pool = Pool(
    shap_sample[features],
    cat_features=cat_indices,
)

shap_values = model.get_feature_importance(
    shap_pool,
    type="ShapValues",
)

mean_abs_shap = np.abs(
    shap_values[:, :-1]
).mean(axis=0)

shap_importance = (
    pd.DataFrame({
        "feature": features,
        "mean_abs_shap_log_price": mean_abs_shap,
    })
    .sort_values(
        "mean_abs_shap_log_price",
        ascending=False,
    )
)

display(shap_importance)

plot_data = shap_importance.head(12).sort_values(
    "mean_abs_shap_log_price"
)

plt.figure(figsize=(9, 6))
plt.barh(
    plot_data["feature"],
    plot_data["mean_abs_shap_log_price"],
)
plt.title("Global CatBoost SHAP importance")
plt.xlabel("Mean |SHAP| contribution to log-price")
plt.tight_layout()
plt.show()


## 11. Failure analysis and regional slices

High headline accuracy is not enough. The model is audited by property type and county, and the largest misses are inspected.


In [ ]:
slice_property = (
    comparison
    .groupby("property_type")
    .agg(
        rows=("actual", "size"),
        MAE=("absolute_error", "mean"),
        MAPE=("absolute_pct_error", "mean"),
    )
    .sort_values("MAE", ascending=False)
)

slice_county = (
    comparison
    .groupby("county")
    .agg(
        rows=("actual", "size"),
        MAE=("absolute_error", "mean"),
        MAPE=("absolute_pct_error", "mean"),
    )
    .query("rows >= 500")
    .sort_values("MAE", ascending=False)
)

display(slice_property.round(3))
display(slice_county.head(15).round(3))

worst_errors = comparison.nlargest(
    12,
    "absolute_error",
)[[
    "actual",
    "prediction",
    "absolute_error",
    "property_type",
    "county",
]]

display(worst_errors)

plot_county = (
    slice_county
    .head(12)
    .sort_values("MAE")
)

plt.figure(figsize=(10, 6))
plt.barh(
    plot_county.index,
    plot_county["MAE"],
)
plt.title("Counties with highest MAE among 500+ test sales")
plt.xlabel("MAE (£)")
plt.tight_layout()
plt.show()


## 12. Provisional July–August 2026 monitoring

The newest HM Land Registry months are not used for headline scoring because registration lags make them incomplete. They are still useful for **monitoring feature mix and registered-price shifts**.

This section deliberately reports descriptive changes rather than claiming true market growth from incomplete volumes.


In [ ]:
if len(provisional):
    historical_reference = test.copy()

    monitoring = pd.DataFrame({
        "window": [
            "Jan-Jun 2026 test",
            "Jul-Aug 2026 provisional",
        ],
        "rows": [
            len(historical_reference),
            len(provisional),
        ],
        "median_price": [
            historical_reference["price"].median(),
            provisional["price"].median(),
        ],
        "mean_price": [
            historical_reference["price"].mean(),
            provisional["price"].mean(),
        ],
        "unique_postcode_districts": [
            historical_reference["postcode_district"].nunique(),
            provisional["postcode_district"].nunique(),
        ],
    })

    display(monitoring.round(2))

    provisional_pred = np.expm1(
        model.predict(
            provisional[features]
        )
    ).clip(
        20_000,
        5_000_000,
    )

    provisional_residual = np.abs(
        provisional["price"].to_numpy()
        - provisional_pred
    )

    print({
        "provisional_rows": len(provisional),
        "provisional_model_MAE_descriptive_only":
            float(provisional_residual.mean()),
    })

    month_monitor = (
        model_df[
            model_df["transfer_date"]
            >= pd.Timestamp("2026-01-01")
        ]
        .assign(
            month=lambda x:
                x["transfer_date"].dt.to_period("M").astype(str)
        )
        .groupby("month")
        .agg(
            rows=("price", "size"),
            median_price=("price", "median"),
        )
        .reset_index()
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 4.5),
    )

    axes[0].bar(
        month_monitor["month"],
        month_monitor["rows"],
    )
    axes[0].set_title(
        "Registered transaction count — recent months are incomplete"
    )
    axes[0].tick_params(
        axis="x",
        rotation=45,
    )

    axes[1].plot(
        month_monitor["month"],
        month_monitor["median_price"],
        marker="o",
    )
    axes[1].set_title(
        "Registered median price by month"
    )
    axes[1].tick_params(
        axis="x",
        rotation=45,
    )

    plt.tight_layout()
    plt.show()
else:
    print(
        "No post-June 2026 rows are present in this source snapshot."
    )


## 13. Deployment profile and exact model reload

In [ ]:
model_path = (
    ROOT
    / "uk_house_price_catboost.cbm"
)

model.save_model(
    str(model_path)
)

reloaded = CatBoostRegressor()
reloaded.load_model(
    str(model_path)
)

reload_sample = test[features].iloc[:500]

original_log = model.predict(
    reload_sample
)

restored_log = reloaded.predict(
    reload_sample
)

reload_delta = float(
    np.max(
        np.abs(
            original_log
            - restored_log
        )
    )
)

inference_sample = test[features].iloc[:10_000]

_ = model.predict(
    inference_sample.iloc[:100]
)

start = time.perf_counter()
_ = model.predict(
    inference_sample
)
elapsed = time.perf_counter() - start

deployment_profile = pd.DataFrame([{
    "rows_profiled": len(inference_sample),
    "total_seconds": elapsed,
    "milliseconds_per_row":
        1000 * elapsed / len(inference_sample),
    "rows_per_second":
        len(inference_sample) / elapsed,
    "serialized_model_MB":
        model_path.stat().st_size / (1024 ** 2),
    "max_reload_delta": reload_delta,
}])

display(deployment_profile.round(6))

assert reload_delta < 1e-12


## 14. Integrity checks and interview evidence

In [ ]:
checks = {
    "real_source_loaded":
        raw_rows > 500_000,
    "transaction_ids_unique":
        int(source_audit["unique_ids"].iloc[0])
        == raw_rows,
    "source_reaches_2026":
        pd.Timestamp(
            source_audit["max_date"].iloc[0]
        ) >= pd.Timestamp("2026-06-30"),
    "future_test_is_jan_to_jun_2026":
        (
            test["transfer_date"].min()
            >= pd.Timestamp("2026-01-01")
            and test["transfer_date"].max()
            <= pd.Timestamp("2026-06-30")
        ),
    "chronological_split":
        (
            train["transfer_date"].max()
            < validation["transfer_date"].min()
            < test["transfer_date"].min()
        ),
    "predictions_finite":
        np.isfinite(pred_test).all(),
    "model_beats_strong_baseline":
        model_mae < baseline_mae,
    "interval_order_valid":
        (
            np.all(lower <= pred_test)
            and np.all(pred_test <= upper)
        ),
    "model_reload_exact":
        reload_delta < 1e-12,
}

display(
    pd.Series(
        checks,
        name="passed",
    ).to_frame()
)

assert all(checks.values())

summary = {
    "source":
        "HM Land Registry Price Paid Data 2025-2026",
    "source_max_date":
        str(
            pd.Timestamp(
                source_audit["max_date"].iloc[0]
            ).date()
        ),
    "raw_transactions":
        raw_rows,
    "model_population":
        accepted_rows,
    "train_rows":
        len(train),
    "validation_rows":
        len(validation),
    "untouched_test_rows":
        len(test),
    "provisional_monitoring_rows":
        len(provisional),
    "catboost_test":
        {
            key: float(value)
            for key, value
            in evaluation.loc["CatBoost"].items()
        },
    "mae_improvement_vs_strong_baseline_pct":
        float(improvement * 100),
    "interval_test_coverage":
        float(covered.mean()),
    "model_reload_delta":
        reload_delta,
}

print("RUN-DERIVED SUMMARY")
print(
    json.dumps(
        summary,
        indent=2,
    )
)

print(
    "\nCV bullet:\n"
    f"Built a leakage-safe England & Wales "
    f"property-price pipeline over {raw_rows:,} "
    f"official HM Land Registry transactions "
    f"using DuckDB, Pandas and CatBoost; "
    f"achieved £{model_mae:,.0f} MAE on "
    f"{len(test):,} untouched Jan-Jun 2026 sales, "
    f"{improvement:.1%} better than a "
    f"postcode-district/property-type baseline, "
    f"with conformal intervals, SHAP diagnostics "
    f"and exact model-reload verification."
)

print(
    "\nAll UK house-price real-data integrity checks passed."
)


## 15. Interview-ready summary and limitations

> **I built a real-data property-price pipeline on HM Land Registry transactions using DuckDB SQL for large-file analytics and CatBoost for mixed categorical/tabular modelling. I used a chronological split, benchmarked against a strong postcode-district × property-type median, calibrated uncertainty only on validation residuals, evaluated on untouched Jan–Jun 2026 transactions, explained the model with SHAP values, audited county/property-type failure slices, monitored provisional July–August data separately, and verified exact model serialization/reload.**

### Limitations
- Price Paid Data records registered transactions, not every market listing.
- Recent months are incomplete because registration can lag completion.
- The dataset does not include bedrooms, floor area, condition, EPC rating or interior quality.
- A production automated valuation model should join EPC/UPRN/property-attribute data under compatible terms and validate address/entity matching.
- These are broad statistical estimates, not professional valuations or investment advice.
